# RegMap — Extended Evaluation (nDCG + group-split-by-control)

Self-contained companion to `notebooks/06_regmap_paper_evaluation.ipynb` and a public mirror of `paper/regmap/eval_extras.py`. It reproduces the two additions reported in the paper:

1. **nDCG@{5,10}** on the original 15% pair-level hold-out (aligns the primary protocol with the nDCG@10 used in the cloud-compliance literature, Bianchi et al., KES 2026).
2. A **stricter group-split-by-control** protocol: the 15% test split is taken over unique `nist_control_id` values (seed 42) so **no control appears in both train and test**, ruling out leakage of control-level phrasing.

Everything is deterministic (seed 42 for split and bootstrap; dense ranking is deterministic given the saved model). Task: given a NIST SP 800-53 control, retrieve the correct HIPAA Security Rule provision(s) from the 60-provision corpus; ground truth is the official NIST crosswalk.

In [1]:
import re, json
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cos
from sklearn.model_selection import train_test_split

def find_root(start):
    for a in [start, *start.parents]:
        if (a / 'data' / 'processed').exists():
            return a
    return start

ROOT = find_root(Path.cwd())
DATA_PROC = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'models' / 'regmap-embedder'
BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
SEED, KS, NDCG_KS, BOOT = 42, [1,3,5,10], [5,10], 1000
rng = np.random.default_rng(SEED)
print('root:', ROOT)

C:\Users\User\ai-cybersecurity-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


root: C:\Users\User\ai-cybersecurity-portfolio


## Ranking backends and metrics (incl. nDCG)

In [2]:
def rank_dense(model, queries, corpus):
    cemb = model.encode(corpus, convert_to_tensor=True, show_progress_bar=False)
    out = []
    for q in queries:
        qe = model.encode(q, convert_to_tensor=True, show_progress_bar=False)
        s = torch.nn.functional.cosine_similarity(qe, cemb)
        out.append(torch.argsort(s, descending=True).tolist())
    return out

def _tok(s): return re.findall(r'[a-z0-9]+', s.lower())
def rank_bm25(queries, corpus):
    bm = BM25Okapi([_tok(c) for c in corpus])
    return [list(np.argsort(bm.get_scores(_tok(q)))[::-1]) for q in queries]
def rank_tfidf(queries, corpus):
    vec = TfidfVectorizer().fit(corpus + queries); C = vec.transform(corpus)
    return [list(np.argsort(sk_cos(vec.transform([q]), C)[0])[::-1]) for q in queries]

def _dcg(rels): return sum(r/np.log2(i+1) for i, r in enumerate(rels, 1))

def per_query(order, rel, ks=KS, ndcg_ks=NDCG_KS):
    rel = set(rel); out = {}
    for k in ks:
        out[f'recall@{k}'] = len(rel & set(order[:k]))/len(rel) if rel else 0.0
    rr = next((1.0/i for i, idx in enumerate(order, 1) if idx in rel), 0.0)
    out['rr'] = rr
    hits = ap = 0.0
    for i, idx in enumerate(order, 1):
        if idx in rel:
            hits += 1; ap += hits/i
    out['ap'] = ap/len(rel) if rel else 0.0
    for k in ndcg_ks:
        gains = [1.0 if idx in rel else 0.0 for idx in order[:k]]
        idcg = _dcg([1.0]*min(k, len(rel))) if rel else 0.0
        out[f'ndcg@{k}'] = (_dcg(gains)/idcg) if idcg > 0 else 0.0
    return out

def aggregate(per_q, keys):
    arr = {k: np.array([m[k] for m in per_q]) for k in keys}
    n = len(per_q); res = {}
    for k, v in arr.items():
        boot = np.array([v[rng.integers(0, n, n)].mean() for _ in range(BOOT)])
        res[k] = (float(v.mean()), float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5)))
    return res

def evaluate(orders, rel_per_q):
    keys = [f'recall@{k}' for k in KS] + ['rr','ap'] + [f'ndcg@{k}' for k in NDCG_KS]
    return aggregate([per_query(o, r) for o, r in zip(orders, rel_per_q)], keys)

## Load crosswalk and build corpus / relevance

In [3]:
cw = pd.read_csv(DATA_PROC / 'labeled_pairs.csv').dropna(subset=['nist_text','hipaa_text'])
corpus = list(cw['hipaa_text'].unique())
cidx = {t: i for i, t in enumerate(corpus)}
control_rel = defaultdict(set)
for _, r in cw.iterrows():
    control_rel[r['nist_control_id']].add(cidx[r['hipaa_text']])
print(f'{len(cw)} pairs | {cw.nist_control_id.nunique()} controls | corpus {len(corpus)} provisions')

def build_orders(q_texts):
    orders = {'fine_tuned': rank_dense(SentenceTransformer(str(MODEL_DIR)), q_texts, corpus),
              'base_minilm': rank_dense(SentenceTransformer(BASE_MODEL), q_texts, corpus),
              'bm25': rank_bm25(q_texts, corpus),
              'tfidf': rank_tfidf(q_texts, corpus)}
    return orders

LABELS = {'fine_tuned':'Fine-tuned SBERT','base_minilm':'Base MiniLM','tfidf':'TF-IDF','bm25':'BM25'}

222 pairs | 133 controls | corpus 60 provisions


## 1. nDCG on the original pair-split (single-relevant)
Same 15% pair-level hold-out as the base evaluation; we add nDCG@{5,10}.

In [4]:
_, test = train_test_split(cw, test_size=0.15, random_state=SEED)
q = test['nist_text'].tolist()
rel_single = [{cidx[t]} for t in test['hipaa_text'].tolist()]
orders = build_orders(q)
rows = []
for name in ['fine_tuned','base_minilm','tfidf','bm25']:
    r = evaluate(orders[name], rel_single)
    rows.append({'method': LABELS[name], 'nDCG@5': round(r['ndcg@5'][0],3), 'nDCG@10': round(r['ndcg@10'][0],3)})
pd.DataFrame(rows).set_index('method')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5828.17it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5993.53it/s]

,nDCG@5,nDCG@10
method,,
Fine-tuned SBERT,0.513,0.544
Base MiniLM,0.365,0.434
TF-IDF,0.377,0.412
BM25,0.234,0.273


## 2. Stricter group-split-by-control (single-relevant)
The 15% split is taken over unique control IDs, so no control is shared between train and test. This is the leakage-free robustness check reported in the paper.

In [5]:
controls = sorted(cw['nist_control_id'].unique())
_, test_controls = train_test_split(controls, test_size=0.15, random_state=SEED)
test_controls = set(test_controls)
gt = cw[cw['nist_control_id'].isin(test_controls)]
gq = gt['nist_text'].tolist()
g_single = [{cidx[t]} for t in gt['hipaa_text'].tolist()]
print(f'{len(gq)} test queries from {len(test_controls)} held-out controls')
gorders = build_orders(gq)
rows = []
for name in ['fine_tuned','base_minilm','tfidf','bm25']:
    r = evaluate(gorders[name], g_single)
    rows.append({'method': LABELS[name],
                 'R@1': round(r['recall@1'][0],3), 'R@3': round(r['recall@3'][0],3),
                 'R@5': round(r['recall@5'][0],3), 'R@10': round(r['recall@10'][0],3),
                 'MRR': round(r['rr'][0],3), 'MAP': round(r['ap'][0],3),
                 'nDCG@10': round(r['ndcg@10'][0],3)})
ft = evaluate(gorders['fine_tuned'], g_single)['recall@5']
print(f"fine-tuned group-split Recall@5 = {ft[0]:.3f} (95% CI {ft[1]:.3f}-{ft[2]:.3f})")
pd.DataFrame(rows).set_index('method')

33 test queries from 20 held-out controls


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5142.53it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6010.54it/s]

fine-tuned group-split Recall@5 = 0.848 (95% CI 0.727-0.970)


,R@1,R@3,R@5,R@10,MRR,MAP,nDCG@10
method,,,,,,,
Fine-tuned SBERT,0.424,0.727,0.848,0.939,0.597,0.597,0.677
Base MiniLM,0.273,0.455,0.545,0.697,0.404,0.404,0.459
TF-IDF,0.212,0.333,0.424,0.485,0.313,0.313,0.336
BM25,0.182,0.364,0.455,0.515,0.303,0.303,0.339


**Takeaway.** Under the stricter, leakage-free group split the fine-tuning gain is undiminished (single-relevant Recall@5 ~0.85 vs ~0.55 for the base encoder and <0.46 for the lexical baselines), indicating the model has learned framework-level semantic correspondence rather than memorized individual control phrasings. These numbers reproduce Table 1 (nDCG column) and the group-split table in `paper/regmap/paper.tex`.